# Step 8c (Kaggle CPU) — DFT relaxation trajectories of BaTiO3 and PbTiO3 for fine-tuning
The MLIPs are trained on PBE (most models) or r2SCAN. To separate the error of the machine-learning model from the error of the DFT functional, we compute the same quantities with DFT ourselves.

**Run on Kaggle:** Settings (right panel) → Internet **On** (needs phone verification), Accelerator **None (CPU)**. Then click **Save Version → Save & Run All (Commit)**. The run continues on Kaggle's servers even if you close the browser (max 12 h). When it is finished, open the version → **Output** tab → download `train_relaxtraj.extxyz` (all ionic steps of the vc-relax runs, with energies, forces and stresses).


In [ ]:
# 1. Install Quantum ESPRESSO from conda-forge (3-5 min)
import os, subprocess
os.makedirs('/tmp/k', exist_ok=True)
if not os.path.exists('/tmp/k/qe/bin/pw.x'):
    !wget -qO- https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /tmp/k bin/micromamba
    !/tmp/k/bin/micromamba create -y -q -p /tmp/k/qe -c conda-forge qe > /tmp/k/mamba.log 2>&1; tail -3 /tmp/k/mamba.log
PW = '/tmp/k/qe/bin/pw.x'
assert os.path.exists(PW), 'Quantum ESPRESSO not installed - send the output above to Claude'
# make sure all shared libraries are found (fix soname mismatches, e.g. libelpa)
import glob, re
os.environ['LD_LIBRARY_PATH'] = '/tmp/k/qe/lib:' + os.environ.get('LD_LIBRARY_PATH', '')
os.environ['OMPI_ALLOW_RUN_AS_ROOT'] = '1'; os.environ['OMPI_ALLOW_RUN_AS_ROOT_CONFIRM'] = '1'
def missing_libs():
    out = subprocess.run(f'ldd {PW}', shell=True, capture_output=True, text=True).stdout
    return re.findall(r'(\S+) => not found', out)
for lib in missing_libs():
    stem = lib.split('.so')[0]
    cand = sorted(glob.glob(f'/tmp/k/qe/lib/{stem}*.so*'))
    print('missing', lib, '-> candidates', [os.path.basename(c) for c in cand])
    if cand: os.symlink(cand[-1], f'/tmp/k/qe/lib/{lib}')
if missing_libs():
    print('installing missing libraries:', missing_libs())
    subprocess.run('/tmp/k/bin/micromamba install -y -q -p /tmp/k/qe -c conda-forge elpa scalapack > /tmp/k/mamba2.log 2>&1', shell=True)
    for lib in missing_libs():
        cand = sorted(glob.glob(f"/tmp/k/qe/lib/{lib.split('.so')[0]}*.so*"))
        if cand: os.symlink(cand[-1], f'/tmp/k/qe/lib/{lib}')
print('still missing:', missing_libs())
assert not missing_libs(), 'Some libraries are still missing - send this output to Claude'
!pip install -q "ase>=3.23" spglib
MPI = None  # serial + OpenMP: MPI was about 4x slower on Kaggle
print('pw.x =', PW, '| mpirun =', MPI, '| CPUs =', os.cpu_count())


In [ ]:
# 2. Pseudopotentials (GBRV, fallback PseudoDojo)
import os, glob, subprocess, re
OUT = '/kaggle/working'  # only results here; os.makedirs(OUT, exist_ok=True)
SOURCES = {
 'gbrv': ('https://www.physics.rutgers.edu/gbrv/all_{xc}_UPF_v1.5.tar.gz', dict(ecutwfc=45, ecutrho=360)),
 'dojo': ('http://www.pseudo-dojo.org/pseudos/nc-sr-04_{xc}_standard_upf.tgz', dict(ecutwfc=85, ecutrho=340)),
}
PSEUDO = {}
def find_upf(d, el):
    pat = re.compile(rf'^{el}([_.-]|$)', re.I)
    return sorted(f for f in glob.glob(d + '/**/*', recursive=True)
                  if f.lower().endswith('.upf') and pat.match(os.path.basename(f)))
for xc in ['pbe']:
    for src, (url, cut) in SOURCES.items():
        d = f'/tmp/k/pseudo_{xc}_{src}'
        if not os.path.isdir(d) or not glob.glob(d + '/**/*.*', recursive=True):
            os.makedirs(d, exist_ok=True)
            r = subprocess.run(f'wget -q --tries=3 --timeout=60 "{url.format(xc=xc)}" -O /tmp/k/{xc}_{src}.tgz && tar xzf /tmp/k/{xc}_{src}.tgz -C {d}', shell=True)
        ok = all(find_upf(d, el) for el in ['Ba','Pb','Ti','O'])
        print(f'{xc:6s} {src}: download {"OK" if ok else "FAILED"}')
        if ok: PSEUDO[xc] = (d, cut, src); break
print('using:', {k: v[2] for k, v in PSEUDO.items()})
assert 'pbe' in PSEUDO, 'No PBE pseudopotentials could be downloaded - send this output to Claude'


In [ ]:
# 3. Structures and DFT relaxation
import numpy as np, pandas as pd, time, spglib
from ase import Atoms
from ase.io import write, read

XCS = ['pbe']            # add 'pbesol' later
SYSTEMS = {"PbTiO3": ("Pb","Ti"), "BaTiO3": ("Ba","Ti")}
DIRS = {"P4mm": [0,0,1], "R3m": [1,1,1], "Amm2": [1,1,0]}
NP = os.cpu_count()

def perovskite(A, B, a=4.0):
    frac = [[0,0,0],[.5,.5,.5],[.5,.5,0],[.5,0,.5],[0,.5,.5]]
    return Atoms([A,B,"O","O","O"], scaled_positions=frac, cell=np.eye(3)*a, pbc=True)

def distort(at, d, dB=0.10, dO=-0.06):
    at = at.copy(); d = np.array(d, float); d /= np.linalg.norm(d)
    p = at.get_positions(); p[1] += dB*d; p[2:] += dO*d; at.set_positions(p); return at

def spg(at, prec=1e-3):
    return spglib.get_spacegroup((at.cell[:], at.get_scaled_positions(), at.numbers), symprec=prec)

def pseudo(el, xc):
    f = find_upf(PSEUDO[xc][0], el)
    assert f, f'no pseudopotential for {el} {xc}'
    return f[-1]

def run_dft(at, xc, tag):
    wd = f'/tmp/k/run/{xc}/{tag}'; os.makedirs(wd, exist_ok=True)
    pp = {el: os.path.basename(pseudo(el, xc)) for el in set(at.get_chemical_symbols())}
    pdir = os.path.dirname(pseudo('O', xc))
    inp = dict(control=dict(calculation='vc-relax', prefix='x', pseudo_dir=pdir, outdir=wd,
                            etot_conv_thr=1e-6, forc_conv_thr=1e-4, nstep=200),
               system=dict(occupations='fixed', **PSEUDO[xc][1]),
               electrons=dict(conv_thr=1e-9, mixing_beta=0.5),
               ions=dict(ion_dynamics='bfgs'), cell=dict(cell_dynamics='bfgs', press_conv_thr=0.2))
    write(f'{wd}/pw.in', at, format='espresso-in', input_data=inp, pseudopotentials=pp, kpts=(6,6,6))
    t0 = time.time()
    cmd = (f'{MPI} -np {NP} --oversubscribe {PW} -nk {NP} -in pw.in' if MPI else f'OMP_NUM_THREADS={NP} {PW} -in pw.in')
    subprocess.run(f'cd {wd} && {cmd} > pw.out 2>&1', shell=True)
    if 'JOB DONE' not in open(f'{wd}/pw.out').read() and MPI:   # fall back to serial
        subprocess.run(f'cd {wd} && OMP_NUM_THREADS={NP} {PW} -in pw.in > pw.out 2>&1', shell=True)
    txt = open(f'{wd}/pw.out').read()
    fin = read(f'{wd}/pw.out', format='espresso-out', index=-1)
    return fin, ('End final coordinates' in txt or 'bfgs converged' in txt), time.time() - t0

csv = f'{OUT}/dft_relax_summary.csv'
# resume: if a previous dft_results.csv was attached as input (Add Input -> Upload), start from it
prev = glob.glob('/kaggle/input/**/dft_relax_summary*.csv', recursive=True)
if not os.path.exists(csv) and prev:
    import shutil; shutil.copy(prev[0], csv); print('resuming from', prev[0])
done = pd.read_csv(csv) if os.path.exists(csv) else pd.DataFrame()
for xc in XCS:
    for name, (A, B) in SYSTEMS.items():
        for ph, d in DIRS.items():
            if len(done) and ((done.xc == xc) & (done.material == name) & (done.phase == ph)).any():
                continue
            a0 = {"BaTiO3": 4.027, "PbTiO3": 3.966}[name]
            start = distort(perovskite(A, B, a0), d, dB=0.05, dO=-0.03)
            try:
                at, conv, dt = run_dft(start, xc, f'{name}_{ph}')
            except Exception as e:
                print('FAILED', xc, name, ph, repr(e)[:300])
                po = f'/tmp/k/run/{xc}/{name}_{ph}/pw.out'
                tail = ''.join(open(po).readlines()[-20:]) if os.path.exists(po) else 'no pw.out'
                raise RuntimeError('DFT run failed - send this message to Claude:\n' + repr(e)[:300] + '\n' + tail)
            frames = read(f"/tmp/k/run/{xc}/{name}_{ph}/pw.out", format="espresso-out", index=":")
            TRJ = f"{OUT}/train_relaxtraj.extxyz"
            for n, fr in enumerate(frames):
                o = fr.copy(); o.calc = None
                o.info.update(tag=f"relax_{name}_{ph}_{n:03d}", material=name, REF_energy=fr.get_potential_energy(),
                              REF_stress=fr.get_stress(voigt=False).reshape(-1))
                o.arrays["REF_forces"] = fr.get_forces()
                write(TRJ, o, append=True, format="extxyz")
            print(f"   saved {len(frames)} frames for {name} {ph}", flush=True)
            cp = at.cell.cellpar()
            offc = np.linalg.norm(at.get_positions()[1] - (at.get_positions()[0] + at.cell.sum(0)/2))
            row = dict(xc=xc, pseudo=PSEUDO[xc][2], material=name, phase=ph, E_eV=at.get_potential_energy(), a=cp[0], b=cp[1], c=cp[2],
                       alpha=cp[3], beta=cp[4], gamma=cp[5], c_over_a=cp[2]/cp[0], volume=at.get_volume(),
                       B_offcentre_A=offc, spg_final=spg(at), converged=conv, time_s=round(dt))
            done = pd.concat([done, pd.DataFrame([row])], ignore_index=True); done.to_csv(csv, index=False)
            print(f"{xc} {name:7s} {ph:6s} E={row['E_eV']:.4f} c/a={row['c_over_a']:.4f} {row['spg_final']} conv={conv} {dt:.0f}s")


In [ ]:
# 4. Energies relative to cubic, and download
assert len(done), 'No DFT result yet - send the output of cell 3 to Claude'
E0 = done[done.phase == 'Pm-3m'].set_index(['xc','material']).E_eV
done['dE_meV_per_fu'] = [1000*(r.E_eV - E0.get((r.xc, r.material), np.nan)) for r in done.itertuples()]
done.to_csv(csv, index=False)
print('Results in', csv, '- download it from the Output tab')
done.pivot_table(index=['material','phase'], columns='xc', values=['dE_meV_per_fu','c_over_a']).round(3)
